# Exp 17 — EDA синонимов: анализ выхода `05_generate_synonyms.py`

**Что смотрим:**
1. Покрытие: сколько датасетов имеют synonyms.json, сколько колонок/значений покрыто
2. Примеры синонимов колонок
3. Примеры синонимов значений (categorical)
4. Качество: уникальность синонимов, нет ли коллизий с оригиналом
5. Итоговая сводная таблица

In [ ]:
from __future__ import annotations
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({"figure.dpi": 120, "font.size": 11})

ROOT = Path("../..").resolve()
RAW_RU = ROOT / "data" / "raw_ru"

DATASETS = ["lamoda", "cars_ru", "auto_ru", "auto_ru_2020", "ozon", "devices"]

def load_synonyms(name: str) -> dict | None:
    p = RAW_RU / name / "synonyms.json"
    if not p.exists():
        return None
    return json.loads(p.read_text())

def load_parquet(name: str) -> pd.DataFrame | None:
    p = RAW_RU / name / "clean.parquet"
    if not p.exists():
        return None
    return pd.read_parquet(p)

syns = {n: s for n in DATASETS if (s := load_synonyms(n)) is not None}
dfs  = {n: load_parquet(n) for n in syns}

print(f"Датасеты с synonyms.json: {list(syns.keys())}")
print(f"Без синонимов:            {[n for n in DATASETS if n not in syns]}")

---
## 1. Покрытие синонимов

In [ ]:
rows = []
for name, s in syns.items():
    col_syns = s.get("columns", {})
    val_syns = s.get("values", {})

    df = dfs.get(name)
    total_cols = len([c for c in df.columns if c != "id"]) if df is not None else "?"

    # Колонки с хотя бы одним синонимом
    cols_with_syns = sum(1 for v in col_syns.values() if v)
    avg_col_syns = np.mean([len(v) for v in col_syns.values() if v]) if cols_with_syns else 0

    # Покрытие значений
    total_val_cols = len(val_syns)
    val_cols_nonempty = sum(1 for vmap in val_syns.values() if any(v for v in vmap.values()))
    total_values = sum(len(vmap) for vmap in val_syns.values())
    values_with_syns = sum(
        sum(1 for v in vmap.values() if v)
        for vmap in val_syns.values()
    )

    rows.append({
        "датасет": name,
        "колонок всего": total_cols,
        "колонок в col_syns": len(col_syns),
        "с ≥1 синонимом": cols_with_syns,
        "покрытие колонок": f"{cols_with_syns/len(col_syns):.0%}" if col_syns else "—",
        "ср. синонимов/кол": round(avg_col_syns, 1),
        "cat-колонок в val_syns": total_val_cols,
        "значений покрыто": f"{values_with_syns}/{total_values}",
        "% значений с syn": f"{values_with_syns/total_values:.0%}" if total_values else "—",
    })

cov = pd.DataFrame(rows).set_index("датасет")
display(cov)

In [ ]:
# Визуализация покрытия
names = list(syns.keys())
n_total = []
n_covered = []
for name in names:
    s = syns[name]
    col_syns = s.get("columns", {})
    n_total.append(len(col_syns))
    n_covered.append(sum(1 for v in col_syns.values() if v))

x = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(x, n_covered, label="с синонимами", color="steelblue")
axes[0].bar(x, [t - c for t, c in zip(n_total, n_covered)],
            bottom=n_covered, label="без синонимов", color="salmon", alpha=0.7)
axes[0].set_xticks(x); axes[0].set_xticklabels(names, rotation=20)
axes[0].set_title("Покрытие имён колонок")
axes[0].legend()

# Среднее число синонимов на колонку
avg_syns = [
    np.mean([len(v) for v in syns[n].get("columns", {}).values() if v]) if syns[n].get("columns") else 0
    for n in names
]
axes[1].bar(names, avg_syns, color="steelblue")
axes[1].set_xticklabels(names, rotation=20)
axes[1].set_title("Среднее число синонимов на колонку")
axes[1].axhline(3, color="red", linestyle="--", label="target=3")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. Примеры синонимов колонок

In [ ]:
N_SHOW = 10  # колонок на датасет

for name, s in syns.items():
    col_syns = s.get("columns", {})
    if not col_syns:
        print(f"[{name}] нет синонимов колонок\n")
        continue

    print(f"\n{'='*60}")
    print(f"{name.upper()} — синонимы имён колонок (топ {N_SHOW})")
    print(f"{'='*60}")

    # Сортируем: сначала с синонимами, потом без
    items = sorted(col_syns.items(), key=lambda kv: -len(kv[1]))
    for col, slist in items[:N_SHOW]:
        syn_str = " | ".join(f'"{s}"' for s in slist) if slist else "(нет)"
        print(f"  {col:<35} → {syn_str}")

---
## 3. Примеры синонимов значений

In [ ]:
N_COLS_SHOW = 4   # categorical колонок на датасет
N_VALS_SHOW = 6   # значений на колонку

for name, s in syns.items():
    val_syns = s.get("values", {})
    if not val_syns:
        print(f"[{name}] нет синонимов значений\n")
        continue

    print(f"\n{'='*60}")
    print(f"{name.upper()} — синонимы значений")
    print(f"{'='*60}")

    # Показываем колонки с наибольшим покрытием значений
    col_coverage = {col: sum(1 for v in vmap.values() if v) / len(vmap)
                    for col, vmap in val_syns.items() if vmap}
    top_cols = sorted(col_coverage, key=col_coverage.get, reverse=True)[:N_COLS_SHOW]

    for col in top_cols:
        vmap = val_syns[col]
        covered = sum(1 for v in vmap.values() if v)
        print(f"\n  Колонка: {col} ({covered}/{len(vmap)} значений)")
        for val, slist in list(vmap.items())[:N_VALS_SHOW]:
            syn_str = " | ".join(f'"{s}"' for s in slist) if slist else "(нет)"
            print(f"    {str(val):<25} → {syn_str}")

---
## 4. Качество синонимов

In [ ]:
quality_rows = []

for name, s in syns.items():
    col_syns = s.get("columns", {})
    val_syns = s.get("values", {})

    # Колонки без синонимов
    no_col_syn = [c for c, v in col_syns.items() if not v]

    # Синонимы-дубликаты оригинала в именах колонок
    col_self_dup = [c for c, v in col_syns.items() if c in v]

    # Пустые синонимы значений
    vals_no_syn = sum(
        sum(1 for sv in vmap.values() if not sv)
        for vmap in val_syns.values()
    )
    vals_total = sum(len(vmap) for vmap in val_syns.values())

    # Синонимы значений == оригинал (LLM вернул само значение)
    val_self_dup = sum(
        sum(1 for orig_val, sv in vmap.items() if str(orig_val) in sv)
        for vmap in val_syns.values()
    )

    quality_rows.append({
        "датасет": name,
        "колонок без syn": len(no_col_syn),
        "col_syn == оригинал": len(col_self_dup),
        "значений без syn": vals_no_syn,
        "% значений без syn": f"{vals_no_syn/vals_total:.0%}" if vals_total else "—",
        "val_syn == оригинал": val_self_dup,
        "примеры col без syn": no_col_syn[:4],
    })

q_df = pd.DataFrame(quality_rows).set_index("датасет")
display(q_df.drop(columns=["примеры col без syn"]))

print()
for _, r in q_df.iterrows():
    if r["примеры col без syn"]:
        print(f"{r.name}: колонки без синонимов → {r['примеры col без syn']}")

In [ ]:
# Распределение длин синонимов (насколько они разнообразны)
fig, axes = plt.subplots(1, len(syns), figsize=(4 * len(syns), 3), sharey=False)
if len(syns) == 1:
    axes = [axes]

for ax, (name, s) in zip(axes, syns.items()):
    col_syns = s.get("columns", {})
    lens = [len(v) for v in col_syns.values() if v]
    if lens:
        ax.hist(lens, bins=range(0, max(lens) + 2), color="steelblue", edgecolor="white", rwidth=0.8)
    ax.set_title(f"{name}\n(медиана={int(np.median(lens)) if lens else 0})", fontsize=10)
    ax.set_xlabel("синонимов у колонки")

plt.suptitle("Распределение числа синонимов на имя колонки", y=1.02)
plt.tight_layout()
plt.show()

---
## 5. Разнообразие синонимов: лингвистический анализ

Смотрим:
- Языковой микс (рус/анг синонимы)
- Нотационные различия (snake_case vs CamelCase vs пробелы)
- Длина синонимов

In [ ]:
import unicodedata

def detect_lang(text: str) -> str:
    """Грубое определение языка по символам."""
    cyr = sum(1 for c in text if unicodedata.category(c) == "Ll" and c.lower() in "абвгдеёжзийклмнопрстуфхцчшщъыьэюяabcdefghijklmnopqrstuvwxyz")
    n_cyr = sum(1 for c in text if "а" <= c.lower() <= "я" or c.lower() == "ё")
    n_lat = sum(1 for c in text if "a" <= c.lower() <= "z")
    if n_cyr == 0 and n_lat == 0:
        return "other"
    return "ru" if n_cyr >= n_lat else "en"

def notation(text: str) -> str:
    if "_" in text:
        return "snake_case"
    if text != text.lower() and text != text.upper() and any(c.isupper() for c in text[1:]):
        return "CamelCase"
    if " " in text:
        return "words"
    return "lower"

rows = []
for name, s in syns.items():
    col_syns = s.get("columns", {})
    all_syns = [sv for v in col_syns.values() for sv in v]
    if not all_syns:
        continue
    ru = sum(1 for t in all_syns if detect_lang(t) == "ru")
    en = sum(1 for t in all_syns if detect_lang(t) == "en")
    notations = pd.Series([notation(t) for t in all_syns]).value_counts()
    rows.append({
        "датасет": name,
        "всего синонимов": len(all_syns),
        "рус": f"{ru/len(all_syns):.0%}",
        "англ": f"{en/len(all_syns):.0%}",
        "snake_case": notations.get("snake_case", 0),
        "CamelCase": notations.get("CamelCase", 0),
        "words": notations.get("words", 0),
        "lower": notations.get("lower", 0),
        "ср.длина": round(np.mean([len(t) for t in all_syns]), 1),
    })

lang_df = pd.DataFrame(rows).set_index("датасет")
display(lang_df)

---
## 6. Итог — сводная таблица

In [ ]:
summary_rows = []
for name in DATASETS:
    s = syns.get(name)
    if s is None:
        summary_rows.append({"датасет": name, "статус": "❌ нет synonyms.json"})
        continue

    col_syns = s.get("columns", {})
    val_syns = s.get("values", {})

    n_col = len(col_syns)
    n_col_ok = sum(1 for v in col_syns.values() if v)
    n_vals = sum(len(vmap) for vmap in val_syns.values())
    n_vals_ok = sum(sum(1 for sv in vmap.values() if sv) for vmap in val_syns.values())
    avg_csyn = np.mean([len(v) for v in col_syns.values() if v]) if n_col_ok else 0

    ok = n_col_ok / n_col >= 0.8 and avg_csyn >= 2 if n_col else False
    status = "✓ OK" if ok else "⚠ частично"

    summary_rows.append({
        "датасет": name,
        "статус": status,
        "col покрытие": f"{n_col_ok}/{n_col}",
        "ср. syn/col": round(avg_csyn, 1),
        "val покрытие": f"{n_vals_ok}/{n_vals}" if n_vals else "—",
        "cat колонок": len(val_syns),
    })

summary = pd.DataFrame(summary_rows).set_index("датасет")
display(summary)
print("\nПримечание: 'OK' = ≥80% колонок покрыто + ср. ≥2 синонима")